# Robust Financial News & Sentiment Scraper (v3)

This notebook provides a robust, production-grade scraper for financial text data, solving previous data desynchronization issues, capturing publication timestamps, and storing outputs in clean **JSON** and **JSONL** formats.

### Pipeline Vision:
1. **Scraping** (This Notebook): Extract Headlines, Full Articles, Timestamps, Source, Category, and URL into structured JSON.
2. **LLM NER**: Extract identified entities (`Company`, `Sector`, `Ticker`).
3. **LLM Interest Labeling**: Filter whether the news is market-relevant (`Interest = 1`) or noise (`0`).
4. **NLP Sentiment Analysis**: Classify sentiment (FinBERT / LLM) and correlate with stock price movements.

### Why JSON / JSONL is Better Than CSV:
- **Zero Escaping Issues**: Financial articles contain quotes, ₹ currency symbols, commas, and dialogue that break CSV parsers.
- **Native LLM Format**: LLM APIs (OpenAI, Claude, Gemini, HuggingFace) natively ingest and return JSON.
- **Structured & Hierarchical**: Allows future expansion (e.g. nested NER entities, comments, confidence scores).

In [2]:
import sys
import os
import json
from pathlib import Path

# Bulletproof dynamic path resolution for scraper module
def resolve_scraper_path():
    start = Path.cwd()
    candidates = [start] + list(start.parents)
    for c in candidates:
        if (c / 'Web Scraping' / 'scraper' / '__init__.py').exists():
            return str(c / 'Web Scraping')
        if (c / 'scraper' / '__init__.py').exists():
            return str(c)
    return None

scraper_root = resolve_scraper_path()
if scraper_root and scraper_root not in sys.path:
    sys.path.insert(0, scraper_root)

import pandas as pd
import matplotlib.pyplot as plt
from scraper import MoneycontrolScraper, FinancialRSSScraper, RedditFinancialScraper

print(f"✓ Scraper modules successfully loaded from: {scraper_root}")

✓ Scraper modules successfully loaded from: /mnt/c/Users/arnav/OneDrive/Documents/Self_Apps/Stock-Market-Prediction/Web Scraping


## 1. Moneycontrol News Scraper

Scrapes clean article text, headlines, and exact publication timestamps while filtering out promotional ads (`Remove Ad`) and disclaimers.

In [ ]:
# Initialize Moneycontrol scraper with polite delay
mc_scraper = MoneycontrolScraper(delay_range=(0.5, 1.2))

# Scrape desired categories and page depth
# Available categories: 'companies', 'stocks', 'markets', 'earnings', 'economy'
categories_to_scrape = ['companies', 'stocks']
pages_per_category = 1  # Increase to 3-5+ for larger dataset creation

for category in categories_to_scrape:
    mc_scraper.scrape_category(category=category, max_pages=pages_per_category)

df_moneycontrol = mc_scraper.to_dataframe()
print(f"Total Moneycontrol articles scraped: {len(df_moneycontrol)}")
df_moneycontrol.head()

### Data Integrity Verification
Verify that headlines and articles are 1:1 synchronized with zero null values or promo strings.

In [ ]:
if not df_moneycontrol.empty:
    assert len(df_moneycontrol['Headlines']) == len(df_moneycontrol['Article']), "Mismatch in headlines and article counts!"
    assert not df_moneycontrol['Headlines'].isnull().any(), "Null values found in Headlines!"
    assert not df_moneycontrol['Article'].isnull().any(), "Null values found in Article!"
    assert not df_moneycontrol['Headlines'].str.lower().str.contains('remove ad').any(), "Promotional ad leaked into headlines!"
    print("✓ Integrity Check Passed: All articles and headlines are 100% synchronized.")
    print(f"✓ Sample Timestamp: {df_moneycontrol.iloc[0]['Published_At']}")
else:
    print("No articles collected yet.")

## 2. Complementary Sources (RSS & Social Sentiment)

Collect additional financial news via Google News RSS or retail trader psychology via Reddit (`r/IndianStreetBets`).

In [ ]:
# 2A. Google News Financial RSS
rss_scraper = FinancialRSSScraper()
rss_items = rss_scraper.fetch_feed(query="when:7d site:moneycontrol.com/news/business", max_items=10)
df_rss = rss_scraper.to_dataframe()

# 2B. Reddit Financial Sentiment (r/IndianStreetBets)
reddit_scraper = RedditFinancialScraper()
reddit_posts = reddit_scraper.fetch_subreddit_feed(subreddit="IndianStreetBets", feed="hot", max_posts=10)
df_reddit = reddit_scraper.to_dataframe()

print(f"Scraped {len(df_rss)} items from RSS.")
print(f"Scraped {len(df_reddit)} posts from Reddit.")

## 3. Merge Datasets & Exploratory Analysis

In [ ]:
# Combine all collected datasets into unified DataFrame
df_combined = pd.concat([df_moneycontrol, df_rss, df_reddit], ignore_index=True)
print(f"Total Combined Dataset Size: {len(df_combined)} rows")
print("\nSource Breakdown:")
print(df_combined['Source'].value_counts())

# Calculate word counts
df_combined['Word_Count'] = df_combined['Article'].apply(lambda x: len(str(x).split()))

# Visualize word count distribution
plt.figure(figsize=(9, 4))
plt.hist(df_combined['Word_Count'], bins=20, color='#1e56a0', edgecolor='black', alpha=0.85)
plt.title('Article Word Count Distribution')
plt.xlabel('Word Count')
plt.ylabel('Count')
plt.grid(axis='y', alpha=0.3)
plt.show()

## 4. Export to JSON, JSONL & CSV

We save the dataset in both **JSON** (human-readable / structured) and **JSONL** (streaming / LLM fine-tuning and prompting).

In [ ]:
# Determine Datasets output directory
datasets_dir = None
for p in [Path.cwd(), Path.cwd().parent, Path.cwd().parent.parent]:
    if (p / 'Datasets').exists():
        datasets_dir = p / 'Datasets'
        break

if not datasets_dir:
    datasets_dir = Path.cwd() / 'Datasets'
    datasets_dir.mkdir(exist_ok=True)

# 1. Save to JSON (Pretty indented)
json_path = datasets_dir / 'scraped_financial_news.json'
records = df_combined.to_dict(orient='records')
with open(json_path, 'w', encoding='utf-8') as f:
    json.dump(records, f, ensure_ascii=False, indent=2)
print(f"✓ Saved JSON to:  {json_path}")

# 2. Save to JSONL (Line-by-line, ideal for LLM processing)
jsonl_path = datasets_dir / 'scraped_financial_news.jsonl'
with open(jsonl_path, 'w', encoding='utf-8') as f:
    for r in records:
        f.write(json.dumps(r, ensure_ascii=False) + '\n')
print(f"✓ Saved JSONL to: {jsonl_path}")

# 3. Save to CSV for legacy compatibility
csv_path = datasets_dir / 'scraped_financial_news.csv'
df_combined.to_csv(csv_path, index=False, encoding='utf-8')
print(f"✓ Saved CSV to:   {csv_path}")

### Loading the JSON Data for Downstream LLMs
Here is how simple it is to load the JSON in Python for your LLM NER and Sentiment scripts:

In [ ]:
# Load and inspect one record
with open(json_path, 'r', encoding='utf-8') as f:
    data = json.load(f)

print(f"Total loaded articles: {len(data)}")
print("\nSample JSON Record:")
print(json.dumps(data[0], indent=2)[:400] + "...\n}")